In [4]:
import requests
from bs4 import BeautifulSoup, Comment
import pandas as pd
import os
import import_ipynb
from RosterSplits import RosterAndSplits
class SGameLog:
    def __init__(self, team, year, output_folder="output"):
        self.team = team
        self.year = year
        self.output_folder = output_folder
        self.urls = []
        self.all_rows = dict()
        self.all_cols = dict()
    def generate_urls(self):
        self.urls.append("https://www.pro-football-reference.com/teams/" + self.team + "/" + str(self.year) + ".htm")
        self.urls.append("https://www.pro-football-reference.com/teams/" + self.team + "/" + str(self.year) + "/gamelog")
    def evaluate(self):
        self.generate_urls()
        for url in self.urls:
            table_ids, hidden_tables, text = RosterAndSplits.all_tables(url)
            RosterAndSplits.rows_and_columns(self.all_rows, self.all_cols, text, table_ids, hidden_tables)
        for id in self.all_rows:
            flag = ["passing", "team_td_log", "opp_td_log"]
            bool = False
            if id in flag:
                bool = True
            headers = []
            act = []
            data = []
            rows = self.all_rows[id]
            cols = self.all_cols[id]
            for row in rows:
                cols = row.find_all(['td', 'th'])
                cols = [col.text.strip() for col in cols]
                data.append(cols)
            if (bool):
                headers = data[0]
            else:
                headers = data[1]
            duplicates = RosterAndSplits.find_duplicates(headers)
            if (len(duplicates) > 0):
                for item in duplicates:
                    count = duplicates[item][0]
                    all_duplicates = duplicates[item][1]
                    cur_word = item
                    for i in range(count):
                        idx = all_duplicates[i]
                        cur_word += 'R'
                        headers[idx] = cur_word
            df = pd.DataFrame(data, columns=[headers])  # Replace with actual column names
            if bool:
                df = df.drop([0])
            else:
                df = df.drop([0,1])
            df = df.reset_index(drop=True)
            folder_path = self.output_folder
            file_name = id + '1.csv'
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)
            path = os.path.join(folder_path, file_name)
            df.to_csv(path, index=False)

            

In [8]:
#test
gamelog_2004 = SGameLog("clt", "2004", "Stats1")

In [9]:
gamelog_2004.evaluate()

In [10]:
#test2
gamelog_2024 = SGameLog("clt", "2024", "Stats2")

In [11]:
gamelog_2024.evaluate()